In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

import pertpy as pt
import spVIPESmulti

np.random.seed(42)
torch.manual_seed(42)
sc.settings.set_figure_params(dpi=100, frameon=False)
torch.cuda.is_available()

# torch.set_float32_matmul_precision("medium")

In [ ]:
adata = pt.data.kang_2018()

print(f"Shape           : {adata.shape}")
print(f"Cell types      : {adata.obs['cell_type'].nunique()}")
print()

In [ ]:
adata = adata[adata.obs["cell_type"] != "Megakaryocytes"]

In [ ]:
adata.obs.columns

In [ ]:
adata.obs["replicate"].value_counts()

In [ ]:
adata = spVIPESmulti.utils.highly_variable_genes_union(
    adata,
    group_key="batch",
    n_top_genes=4000,
    flavor="seurat_v3",
)
print(f"HVG union: {adata.n_vars} genes")

In [ ]:
# Split into per-time-point AnnData objects
conditions = sorted(adata.obs["label"].unique())
adatas_dict = {}
for condition in conditions:
    sub = adata[adata.obs["label"] == condition].copy()
    # Clean up — prepare_adatas doesn't need extra obsm/uns
    sub.uns = {}
    sub.obsm = {}
    sub.layers = {}
    adatas_dict[condition] = sub
    print(f"  {condition}: {sub.shape}")

# Concatenate with spVIPESmulti
adata_spv = spVIPESmulti.data.prepare_adatas(adatas_dict)

In [ ]:
spVIPESmulti.model.spVIPESmulti.setup_anndata(
    adata_spv,
    groups_key="label",
    label_key="cell_type",
    sample_key="replicate",
    condition_key="label",
    donor_key="replicate",
)

In [ ]:
# Model hyperparameters
N_SHARED   = 20
N_PRIVATE  = 10
N_HIDDEN   = 256
DROPOUT    = 0.2
MAX_EPOCHS = 300
BATCH_SIZE = 1024
KL_WARMUP  = 80

model = spVIPESmulti.model.spVIPESmulti(
    adata_spv,
    n_hidden=N_HIDDEN,
    n_dimensions_shared=N_SHARED,
    n_dimensions_private=N_PRIVATE,
    dropout_rate=DROPOUT,
    disentangle_preset="full_bio",
    disentangle_batch_shared_weight=0.0,
    disentangle_donor_shared_weight=0.5,
    disentangle_donor_private_weight=2,
    contrastive_weight=0.5,
    contrastive_temperature=0.1,
    use_nf_prior=False,
    use_batch_norm=True,
    use_layer_norm=False,
    dispersion="gene",
)
print(model)

In [ ]:
?model.train

In [ ]:
# Get group indices from uns (canonical approach matching prepare_adatas output)
group_indices_list = [list(map(int, g)) for g in adata_spv.uns["groups_obs_indices"]]
print("Group sizes:", [len(g) for g in group_indices_list])

# Train with F1 orthogonality instrumentation enabled. Kang does not expose a
# separate technical batch here, so the F4 batch-shared head is intentionally off.
model.train(
    group_indices_list=group_indices_list,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    train_size=0.9,  # Leave 10% for validation
    early_stopping=True,
    n_epochs_kl_warmup=KL_WARMUP,
    compute_orthogonality_metric=True,
    orthogonality_groupby_keys=("condition", "donor"),
    orthogonality_min_cells_per_stratum=16,
    plan_kwargs={
        "lr_scheduler_type": "cosine",
        "lr_patience": 50,
    },
)

In [ ]:
# Multi-panel training history — one panel per tracked metric
fig = spVIPESmulti.pl.training_curves(model)
plt.show()

In [ ]:
latents = model.get_latent_representation(
    group_indices_list=group_indices_list,
    batch_size=BATCH_SIZE,
)
shared_by_group = latents.get("shared_reordered", latents["shared"])
private_by_group = latents.get("private_reordered", latents["private"])

latent_shared = np.zeros((adata_spv.n_obs, N_SHARED), dtype=np.float32)
latent_private = np.zeros((adata_spv.n_obs, N_PRIVATE), dtype=np.float32)
for group_idx, obs_idx in enumerate(group_indices_list):
    latent_shared[np.asarray(obs_idx)] = shared_by_group[group_idx]
    latent_private[np.asarray(obs_idx)] = private_by_group[group_idx]

latent_private_dataset1 = private_by_group[0]
latent_private_dataset2 = private_by_group[1]


def _probe_balanced_accuracy(x, y, seed=42):
    y = np.asarray(y).astype(str)
    classes, counts = np.unique(y, return_counts=True)
    if classes.size < 2:
        return np.nan
    stratify = y if counts.min() >= 2 else None
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.3,
        random_state=seed,
        stratify=stratify,
    )
    clf = LogisticRegression(max_iter=500, class_weight="balanced")
    clf.fit(x_train, y_train)
    return balanced_accuracy_score(y_test, clf.predict(x_test))

probe_rows = []
for latent_name, x in {"shared": latent_shared, "private": latent_private}.items():
    for target_name, obs_key in {
        "condition": "label",
        "donor": "replicate",
        "cell_type": "cell_type",
    }.items():
        probe_rows.append({
            "latent": latent_name,
            "target": target_name,
            "balanced_accuracy": _probe_balanced_accuracy(x, adata_spv.obs[obs_key]),
        })

probe_df = pd.DataFrame(probe_rows)
probe_df.pivot(index="target", columns="latent", values="balanced_accuracy")

In [ ]:
adata_spv.obsm["X_spVIPES_shared"] = latent_shared
sc.pp.neighbors(adata_spv, use_rep="X_spVIPES_shared", key_added="spvipes_shared")
sc.tl.umap(adata_spv, neighbors_key="spvipes_shared", min_dist=1)
adata_spv.obsm["X_umap_shared"] = adata_spv.obsm["X_umap"].copy()

In [ ]:
sc.pl.embedding(adata_spv, basis="X_umap_shared", color=["cell_type", "label", "replicate"])

In [ ]:
adatas_ctrl = adatas_dict["ctrl"]
adatas_stim = adatas_dict["stim"]

adatas_ctrl.obsm["X_spvipes_private"] = private_by_group[conditions.index("ctrl")]
adatas_stim.obsm["X_spvipes_private"] = private_by_group[conditions.index("stim")]

In [ ]:
sc.pp.neighbors(adatas_ctrl, use_rep="X_spvipes_private", key_added="spvipes_private")
sc.tl.umap(adatas_ctrl, neighbors_key="spvipes_private")

sc.pp.neighbors(adatas_stim, use_rep="X_spvipes_private", key_added="spvipes_private")
sc.tl.umap(adatas_stim, neighbors_key="spvipes_private")

In [ ]:
sc.pl.embedding(adatas_ctrl, basis="X_umap", color=["cell_type","replicate"])

In [ ]:
ifn_beta_response_genes = [
    "IFNAR1",
    "IFNAR2",
    "IFNB1",
    "STAT1",
    "STAT2",
    "TYK2",
    "JAK1",
    "IRF9",
    "IRF1",
    "IRF7",
    "IRF3",
    "OAS1",
    "OAS2",
    "OAS3",
    "MX1",
    "MX2",
    "ISG15",
    "RSAD2",
    "IFI27",
    "IFI44",
    "IFI44L",
    "IFIT1",
    "IFIT2",
    "IFIT3",
    "IFIT5",
    "IFITM1",
    "IFITM2",
    "IFITM3",
    "IFI6",
    "IFIH1",
    "IFIT1B",
    "IFITM10",
    "OASL",
    "BST2",
    "IFITM5"
]

In [ ]:
sc.tl.score_genes(adatas_ctrl, gene_list=ifn_beta_response_genes, score_name="ISG_score")

In [ ]:
sc.pl.violin(adatas_ctrl, groupby="label", keys="ISG_score", rotation=45)

In [ ]:
sc.tl.score_genes(adatas_stim, gene_list=ifn_beta_response_genes, score_name="ISG_score")

In [ ]:
adatas_stim.obs["ifnb_responsive"] = adatas_stim.obs["ISG_score"] > 5

In [ ]:
sc.pl.embedding(adatas_stim, basis="X_umap", color=["cell_type","ifnb_responsive"])

In [ ]:
# Top genes per shared latent dimension:
top = spVIPESmulti.utils.get_top_genes(model=model, n_top=10)
print(top[["dim", "pos_genes"]].to_string(index=False))

In [ ]:
# Heatmap of top-5 genes per dimension (requires seaborn):
ax = spVIPESmulti.pl.heatmap_loadings(model=model, n_top=5)

In [ ]:
payload = model.embed(batch_size=2048)

In [ ]:
top_stim = spVIPESmulti.utils.get_top_genes(model=model, group_idx=1, latent_type="private", n_top=50, signed=True)

In [ ]:
top_stim.to_csv("test.csv")

In [ ]:
adatas_stim

In [ ]:
private_dims = list(range(N_PRIVATE))

fig = spVIPESmulti.pl.plot_latent_dims_in_umap(
    adatas_stim,
    obsm_key="X_spvipes_private",
    dims=private_dims,
    ncols=5,
    figsize_per_panel=(3.2, 3.0),
)
fig.suptitle(f"Stim private latent dimensions 0-{N_PRIVATE-1}", y=1.02)
fig.show()

In [ ]:
adatas_stim.obs["ifnb_responsive"] = adatas_stim.obs["ifnb_responsive"].astype("category")

In [ ]:
private_dims = list(range(N_PRIVATE))
ncols = 5
nrows = int(np.ceil(len(private_dims) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(25, 25), squeeze=False)
for dim_idx, ax in zip(private_dims, axes.ravel()):
    spVIPESmulti.pl.factor_violin(
        adatas_stim[adatas_stim.obs["cell_type"] == "CD14+ Monocytes"],
        dim_idx=dim_idx,
        groupby="ifnb_responsive",
        obsm_key="X_spvipes_private",
        ax=ax,
        rotation=90,
        show=False,
    )
    ax.set_title(f"Private {dim_idx}")
    ax.set_xlabel("")

for ax in axes.ravel()[len(private_dims):]:
    ax.set_axis_off()

fig.suptitle(f"Stim private latent factor violins 0-{N_PRIVATE-1}", y=1.02)
fig.tight_layout()
fig.show()

In [ ]:
# Decoder-based traversal of z_shared dimensions
traversal = model.traverse_latent(group_idx=0, n_steps=15)
top_genes = spVIPESmulti.traversal.calculate_differential_vars(traversal, top_n=20)
fig = spVIPESmulti.pl.differential_vars_heatmap(traversal)